# Verdant Drift: the working notebook

This is the lab notebook behind the Verdant Drift postmortem (the companion writeup). If you read that, this is the receipts: the detector that worked, the benchmark it beat, and the investigation that talked me out of shipping the rest.

One heads-up before you clone it. This won't run end to end for you as-is. It needs your own Earth Engine project (because you can't use my compute), and a couple of cells are debugging detours I left in on purpose. Treat it as a readable record of the process, not a turn-key pipeline. 🛰️

The arc, roughly:
- **Setup and detection**, the part that worked
- **Validation against real fires** and a head-to-head against classical dNBR
- **Signature typing**, the extension that broke
- **The diagnostics that caught the flaw** and ended the project

If you only have two minutes, maybe jump down to steps 26 through 29. That's where the project kinda falls apart.

## Setup

Standard Earth Engine boot-up: install libraries, authenticate, point at my project. The only thing to swap if you're poking at this is the project ID.

In [ ]:
# Importy McImportFace
import subprocess
subprocess.run(["pip", "install", "geemap", "geopandas", "earthengine-api", "-q"])

import ee
import geemap
import geopandas as gpd
import json

# Authenticate once per session
ee.Authenticate()
ee.Initialize(project='ee-XXXXXXXXXX')  # replace with your OWN project ID

## 1. Study area

San Luis Obispo plus Santa Barbara counties, dissolved into one geometry. I pulled in Santa Barbara early because the Alamo Fire crosses the county line, and bundling the two to make the "Central Coast".

In [2]:
# Load SLO + Santa Barbara counties and combine into one AOI
counties = ee.FeatureCollection("TIGER/2018/Counties")

slo = counties.filter(ee.Filter.And(
    ee.Filter.eq('STATEFP', '06'),
    ee.Filter.eq('NAME', 'San Luis Obispo')
))

sb = counties.filter(ee.Filter.And(
    ee.Filter.eq('STATEFP', '06'),
    ee.Filter.eq('NAME', 'Santa Barbara')
))

# Union into a single AOI geometry
aoi = slo.merge(sb).geometry().dissolve()

# Quick viz check
Map = geemap.Map(center=[35.0, -120.3], zoom=8)
Map.addLayer(
    slo.merge(sb),
    {'color': 'blue', 'fillColor': '0000FF22'},
    'SLO + Santa Barbara Counties'
)
Map.centerObject(aoi, zoom=8)
Map

Map(center=[34.95262691883632, -120.23436761217657], controls=(WidgetControl(options=['position', 'transparent…

## 2. Load the embeddings, build a baseline

AlphaEarth gives every pixel a 64-number fingerprint per year. The loader pulls a year and mosaics every tile covering the area; the baseline builder averages 2017 to 2019 into one stable "this is normal" reference.

Two lessons are already baked in here. The collection is tiled by UTM zone, so an early version using `.first()` quietly grabbed a single tile and analyzed a thin vertical strip of California. `.mosaic()` is the fix. Also, the image IDs are random hashes with no year attached, so you filter by date, not by a year property.

In [3]:
ALPHA_EARTH = "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"

def load_embedding_year(year):
    """Load and mosaic all tiles covering the AOI for a given year."""
    return (ee.ImageCollection(ALPHA_EARTH)
            .filterBounds(aoi)
            .filterDate(f'{year}-01-01', f'{year}-12-31')
            .mosaic()
            .clip(aoi))

def build_baseline(years):
    """Mean embedding across a list of pre-disturbance years (stable reference)."""
    imgs = ee.ImageCollection([load_embedding_year(y) for y in years])
    return imgs.mean()

# Quick sanity check
test = build_baseline([2017, 2018, 2019]).select('A00').reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=ee.Geometry.Point([-120.6, 35.3]).buffer(1000),
    scale=100
).getInfo()
print("Baseline sanity check:", test)  # expect a real number

Baseline sanity check: {'A00': 0.14531940583036854}


## 3. The core idea: distance from normal

Cosine distance between the baseline fingerprint and a test year. Zero means the pixel still looks like itself; higher means something moved it. That's the entire detector. No labels, no training, just "how far did this pixel drift from its own normal." Here I run it for 2020, the fire year.

In [4]:
def cosine_distance(img_a, img_b):
    """Per-pixel cosine distance: 1 - cos(v_a, v_b). 0 = identical, higher = more change."""
    dot    = img_a.multiply(img_b).reduce(ee.Reducer.sum()).rename('dot')
    norm_a = img_a.pow(2).reduce(ee.Reducer.sum()).sqrt().rename('norm_a')
    norm_b = img_b.pow(2).reduce(ee.Reducer.sum()).sqrt().rename('norm_b')
    cos_sim = dot.divide(norm_a.multiply(norm_b)).rename('cos_sim')
    return ee.Image(1).subtract(cos_sim).rename('cosine_distance')

# Detection setup: 3-year pre-fire baseline vs the 2020 fire year
BASELINE_YEARS = [2017, 2018, 2019]
TEST_YEAR      = 2020

baseline = build_baseline(BASELINE_YEARS)
emb_test = load_embedding_year(TEST_YEAR)

# Distance from each pixel's stable baseline to its 2020 value
change = cosine_distance(baseline, emb_test)
print("Baseline-deviation change image computed ✓")

Baseline-deviation change image computed ✓


## 4. Let the data pick the threshold

Instead of hard-coding "disturbed means distance > X," I read the percentiles of the change distribution and threshold off those. p97 is the working default. `tileScale=4` shows up here and everywhere downstream because the baseline computation is memory-hungry and Earth Engine will otherwise politely refuse to run it.

In [5]:
percentiles = change.reduceRegion(
    reducer=ee.Reducer.percentile([90, 95, 97, 98, 99]),
    geometry=aoi,
    scale=300,          # a bit coarser; distribution shape doesn't need fine resolution
    maxPixels=1e9,
    bestEffort=True,
    tileScale=4         # splits the heavy baseline computation into smaller memory chunks
)
print("Percentile thresholds:", percentiles.getInfo())

Percentile thresholds: {'cosine_distance_p90': 0.04874120914943276, 'cosine_distance_p95': 0.06047803145220581, 'cosine_distance_p97': 0.07213033801568604, 'cosine_distance_p98': 0.08398152852920983, 'cosine_distance_p99': 0.11909638632410262}


## 5. Make the mask

Apply the p97 threshold to get a binary disturbed/stable layer, and keep the continuous distance around as a severity layer.

In [6]:
# Use p97
threshold = percentiles.getNumber('cosine_distance_p97')

# Binary disturbance mask: 1 = disturbed, 0 = stable
disturbance_mask = change.gt(threshold).rename('disturbed')

# Continuous severity layer
severity = change.rename('severity')

# Combined output
output = severity.addBands(disturbance_mask)

print(f"Threshold (p97): {threshold.getInfo():.4f}")
print("Disturbance mask created ✓")

Threshold (p97): 0.0721
Disturbance mask created ✓


## 6. Ground truth: known 2020 fires

Pull the MTBS fire perimeters for 2020 to check the detector against reality. Four came back in the area: Soda, Branch, Pond, and Scorpion.

A gotcha lives here: MTBS stores the ignition date as a Unix millisecond timestamp, not a year, so you filter with a date range, not a calendar-year match. That one cost some time.

In [7]:
frap_all = ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")

start_2020 = ee.Date('2020-01-01').millis()
end_2020   = ee.Date('2021-01-01').millis()

fires_2020 = (frap_all
              .filter(ee.Filter.bounds(aoi))
              .filter(ee.Filter.rangeContains('Ig_Date', start_2020, end_2020)))

print("2020 fires found:", fires_2020.size().getInfo())
print("Names:", fires_2020.aggregate_array('Incid_Name').getInfo())

# Rasterize the union of all 2020 perimeters
fire_raster = (ee.Image(0)
               .paint(fires_2020, 1)
               .rename('fire_perimeter')
               .clip(aoi)
               .unmask(0))

2020 fires found: 4
Names: ['SODA', 'BRANCH', 'POND', 'SCORPION']


## 7. Sanity-check the fire layer

A one-liner confirming the rasterized fire layer actually contains pixels. Boring, but exactly the kind of check that saves an hour of confusion later.

In [8]:
fire_check = fire_raster.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=200,
    bestEffort=True,
    tileScale=4
).getInfo()
print("Fire raster pixel sum:", fire_check)   # should be a non-zero number

Fire raster pixel sum: {'fire_perimeter': 1054}


## 8. Take a look

First real look: severity, the disturbed-pixel mask, and the fire perimeters overlaid.

The perimeters are drawn as vector outlines, not the rasterized version. An earlier attempt painted the raster as a solid fill and turned the entire two-county map black, because `unmask(0)` made every zero visible. `selfMask()` was the fix. The raster is for scoring; the vectors are for looking at.

In [9]:
Map = geemap.Map(center=[35.0, -120.3], zoom=8)

# Severity layer (rescaled to match the lower baseline-deviation distances)
Map.addLayer(
    severity.clip(aoi),
    {'min': 0, 'max': 0.15, 'palette': ['white', 'yellow', 'orange', 'red']},
    'Cosine Distance (Severity)',
    shown=True
)

# Binary disturbance mask (only disturbed pixels show; rest transparent)
Map.addLayer(
    disturbance_mask.selfMask().clip(aoi),
    {'palette': ['#e74c3c']},
    'Disturbed Pixels (p97+)',
    shown=True
)

# Fire perimeters as VECTOR outlines (clean, no filled background)
Map.addLayer(
    fires_2020,
    {'color': '#3498db'},
    'MTBS 2020 Fire Perimeters',
    shown=True
)

# Optional: view the rasterized fire footprint used in scoring
Map.addLayer(fire_raster.selfMask(),
             {'palette': ['#3498db']},
             'Fire Raster (scoring footprint)')

Map.centerObject(aoi, zoom=8)
Map

Map(center=[34.95262691883632, -120.23436761217657], controls=(WidgetControl(options=['position', 'transparent…

## 9. Score it (first pass)

Precision, recall, and IoU (Intersection over Union) against the fire perimeters across a few thresholds. This is the unmasked version, before restricting to natural vegetation, so treat it as a rough first read.

In [10]:
def pixel_count(img, band):
    return (img.select(band)
               .reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi,
                             scale=100, maxPixels=1e9, bestEffort=True,
                             tileScale=4)        # same memory fix here
               .getNumber(band).getInfo())

for key in ['cosine_distance_p97', 'cosine_distance_p98', 'cosine_distance_p99']:
    thr  = percentiles.getNumber(key)
    mask = change.gt(thr).rename('disturbed')

    tp = mask.eq(1).And(fire_raster.eq(1)).rename('tp')
    fp = mask.eq(1).And(fire_raster.eq(0)).rename('fp')
    fn = mask.eq(0).And(fire_raster.eq(1)).rename('fn')

    tp_n, fp_n, fn_n = pixel_count(tp,'tp'), pixel_count(fp,'fp'), pixel_count(fn,'fn')
    prec = tp_n/(tp_n+fp_n) if (tp_n+fp_n) else 0
    rec  = tp_n/(tp_n+fn_n) if (tp_n+fn_n) else 0
    iou  = tp_n/(tp_n+fp_n+fn_n) if (tp_n+fp_n+fn_n) else 0
    print(f"{key}: Precision={prec:.3f}  Recall={rec:.3f}  IoU={iou:.3f}")

cosine_distance_p97: Precision=0.026  Recall=0.507  IoU=0.025
cosine_distance_p98: Precision=0.028  Recall=0.381  IoU=0.027
cosine_distance_p99: Precision=0.028  Recall=0.186  IoU=0.025


## 10. Detour: why is my fire layer empty?

This block is me debugging. At one point the fire filter came back empty, so I dumped every MTBS fire in the area regardless of year to figure out why (the Unix-timestamp issue from step 6). I'm leaving it in, because the messy detective work is half of what real analysis actually looks like.

In [11]:
# Check 1: did the filter actually return anything?
print("Features in frap_fire:", fires_2020.size().getInfo())

# Check 2: what does the fire raster actually look like?
fire_check = fire_raster.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=200,
    bestEffort=True
).getInfo()
print("Fire raster pixel sum:", fire_check)

# Check 3: just print everything in MTBS within SLO county regardless of year
all_slo_fires = (ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")
                 .filter(ee.Filter.bounds(aoi)))

print("All MTBS fires in SLO county:", all_slo_fires.size().getInfo())
print("Names:", all_slo_fires.aggregate_array('Incid_Name').getInfo())
print("Dates:", all_slo_fires.aggregate_array('Ig_Date').getInfo())

Features in frap_fire: 4
Fire raster pixel sum: {'fire_perimeter': 1054}
All MTBS fires in SLO county: 128
Names: ['UNNAMED', 'UNNAMED', 'UNNAMED', 'UNNAMED', 'CHIMNEY', 'UNNAMED', 'UNNAMED', 'CAMP ROBERTS', 'HWY 41', 'BAR-4-L BOYSEN RI', 'CAMPODOCINO', 'TAR SPRINGS WEST', 'JALAMA', 'CHRISTY', 'SAGE/ALISO', 'SANTA CRUZ', 'COLSON', 'CENTRWELL', 'LAZARO VMP', 'DYKE DAVID TALLEY VMP', 'PORTER VMP', 'SHELL PEAK', 'UNNAMED', 'UNNAMED', 'FIFTY-EIGHT', 'LAS PILITAS', 'WHEELER 2', 'UNNAMED', 'UNNAMED', 'SLU-0864', 'RANCHITA', 'SUEY RANCH WEST', 'UNNAMED', 'CAMP', 'UNNAMED', 'UNNAMED', 'PAINT', 'SODA', 'ALISO', 'UNNAMED', 'MARRE', 'UNNAMED', 'OAK HILL', 'UNNAMED', 'WASIOJA', 'SPANISH', 'HALOWEEN', 'AZALEA', 'RANGE', 'LOGAN', 'OGILVY', 'UNNAMED', 'UNNAMED', 'CHIMNEY', 'CACHUMA', 'ZACA', 'BALD', 'GAP', 'JESUSITA', 'TEA', 'LA BREA', 'PILITAS', 'HARRIS', 'CANYON', 'REY', 'ALAMO', 'WHITTIER  CA-LBOR-001770', 'THOMAS', 'CAVE', 'UNNAMED', 'UNNAMED', 'UNNAMED', 'ANTELOPE', 'UNNAMED', 'UNNAMED', 'UNNAME

## 11. Mask to natural vegetation

The method only makes sense on wild vegetation, not farm fields that get tilled and replanted on a schedule (that churn would read as constant fake disturbance). I use USDA CDL to keep only natural-veg classes: forest, shrubland, grass/pasture, and wetlands.

I build the mask by keeping the natural classes rather than listing every crop code to drop, because the CDL `cropland` band is the full classified map, not a yes/no crop flag. Then I re-score inside that mask.

In [12]:
# CDL for the test year; keep only natural-vegetation classes (method's domain)
cdl = (ee.ImageCollection("USDA/NASS/CDL")
       .filterDate('2020-01-01', '2020-12-31').first().select('cropland'))

# Natural vegetation only: forest (141-143), shrubland (152),
# grass/pasture (176), wetlands (190, 195)
wildland = (cdl.eq(141).Or(cdl.eq(142)).Or(cdl.eq(143))
            .Or(cdl.eq(152))
            .Or(cdl.eq(176))
            .Or(cdl.eq(190)).Or(cdl.eq(195)))

def pixel_count(img, band):
    return (img.select(band)
               .updateMask(wildland)                 # only count natural-veg pixels
               .reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi,
                             scale=100, maxPixels=1e9, bestEffort=True, tileScale=4)
               .getNumber(band).getInfo())

for key in ['cosine_distance_p97', 'cosine_distance_p98', 'cosine_distance_p99']:
    thr  = percentiles.getNumber(key)
    mask = change.gt(thr).rename('disturbed')
    tp = mask.eq(1).And(fire_raster.eq(1)).rename('tp')
    fp = mask.eq(1).And(fire_raster.eq(0)).rename('fp')
    fn = mask.eq(0).And(fire_raster.eq(1)).rename('fn')
    tp_n, fp_n, fn_n = pixel_count(tp,'tp'), pixel_count(fp,'fp'), pixel_count(fn,'fn')
    prec = tp_n/(tp_n+fp_n) if (tp_n+fp_n) else 0
    rec  = tp_n/(tp_n+fn_n) if (tp_n+fn_n) else 0
    iou  = tp_n/(tp_n+fp_n+fn_n) if (tp_n+fp_n+fn_n) else 0
    print(f"{key}: Precision={prec:.3f}  Recall={rec:.3f}  IoU={iou:.3f}")

cosine_distance_p97: Precision=0.033  Recall=0.496  IoU=0.032
cosine_distance_p98: Precision=0.039  Recall=0.367  IoU=0.036
cosine_distance_p99: Precision=0.045  Recall=0.171  IoU=0.037


## 12. Recalibrate thresholds inside the mask

Recompute the percentile thresholds using only wildland pixels, so the threshold reflects the distribution I actually care about instead of the whole landscape including farms and towns.

In [13]:
change_wild = change.updateMask(wildland)

percentiles_wild = change_wild.reduceRegion(
    reducer=ee.Reducer.percentile([90, 95, 97, 98, 99]),
    geometry=aoi, scale=300, maxPixels=1e9, bestEffort=True, tileScale=4
)
print("Wildland-only thresholds:", percentiles_wild.getInfo())

# Then re-run the Block 9 sweep, swapping `percentiles` for `percentiles_wild`

Wildland-only thresholds: {'cosine_distance_p90': 0.05172992061246956, 'cosine_distance_p95': 0.061493142681497866, 'cosine_distance_p97': 0.07321830969768252, 'cosine_distance_p98': 0.0829868914774249, 'cosine_distance_p99': 0.10645252453946237}


## 13. Final scoring, wildland-calibrated

The honest version of the numbers: scored inside natural vegetation, using wildland-calibrated thresholds. These are the results I'd stand behind for Phase 1.

In [14]:
# Final wildland sweep using wildland-calibrated thresholds
def pixel_count(img, band):
    return (img.select(band)
               .updateMask(wildland)
               .reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi,
                             scale=100, maxPixels=1e9, bestEffort=True, tileScale=4)
               .getNumber(band).getInfo())

for key in ['cosine_distance_p97', 'cosine_distance_p98', 'cosine_distance_p99']:
    thr  = percentiles_wild.getNumber(key)
    mask = change.gt(thr).rename('disturbed')
    tp = mask.eq(1).And(fire_raster.eq(1)).rename('tp')
    fp = mask.eq(1).And(fire_raster.eq(0)).rename('fp')
    fn = mask.eq(0).And(fire_raster.eq(1)).rename('fn')
    tp_n, fp_n, fn_n = pixel_count(tp,'tp'), pixel_count(fp,'fp'), pixel_count(fn,'fn')
    prec = tp_n/(tp_n+fp_n) if (tp_n+fp_n) else 0
    rec  = tp_n/(tp_n+fn_n) if (tp_n+fn_n) else 0
    iou  = tp_n/(tp_n+fp_n+fn_n) if (tp_n+fp_n+fn_n) else 0
    print(f"{key}: Precision={prec:.3f}  Recall={rec:.3f}  IoU={iou:.3f}")

cosine_distance_p97: Precision=0.034  Recall=0.481  IoU=0.033
cosine_distance_p98: Precision=0.038  Recall=0.374  IoU=0.036
cosine_distance_p99: Precision=0.044  Recall=0.223  IoU=0.038


## 14. The benchmark: classical dNBR

To know if the embeddings are worth anything, I need something to beat. dNBR is the standard burn-severity method. The trick that matters: use anniversary dates (spring to spring, a year apart) so seasonal dry-down cancels out. Mismatched seasons make the whole landscape look burned.

In [15]:
S2 = "COPERNICUS/S2_SR_HARMONIZED"

def mask_s2_clouds(img):
    """Mask clouds/shadows via the Scene Classification Layer (SCL)."""
    scl = img.select('SCL')
    # drop 3=cloud shadow, 8=cloud med, 9=cloud high, 10=cirrus
    good = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(good)

def nbr_composite(start, end):
    """Cloud-masked median NBR over a date window."""
    coll = (ee.ImageCollection(S2)
            .filterBounds(aoi)
            .filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
            .map(mask_s2_clouds))
    return coll.median().normalizedDifference(['B8', 'B12']).rename('NBR').clip(aoi)

# Pre = spring 2020 peak green (before summer fire season)
# Post = after the 2020 fire season
# NOTE: verify none of Soda/Branch/Pond/Scorpion ignited before the pre window closes
nbr_pre  = nbr_composite('2020-04-01', '2020-05-31')
nbr_post = nbr_composite('2021-04-01', '2021-05-31')

dnbr = nbr_pre.subtract(nbr_post).rename('dNBR')
print("Anniversary dNBR computed ✓")

Anniversary dNBR computed ✓


## 15. Look at the benchmark

dNBR rendered, plus a quick visual of which pixels the wildland mask keeps versus drops. Always worth eyeballing the mask to confirm it's keeping what you think it is.

In [16]:
Map = geemap.Map(center=[35.0, -120.3], zoom=8)
Map.addLayer(dnbr.clip(aoi),
             {'min': -0.1, 'max': 0.66, 'palette': ['green', 'yellow', 'orange', 'red']},
             'dNBR (classical burn severity)')
Map.addLayer(fires_2020, {'color': '#3498db'}, 'MTBS 2020 Fire Perimeters')
#Map.addLayer(wildland, {'color': '#00cc00'}, 'Wildland')
Map.addLayer(wildland.selfMask(), {'palette': ['#2ecc71']}, 'Wildland (kept)')
Map.addLayer(wildland.Not().selfMask(), {'palette': ['#e74c3c']}, 'Dropped (ag/urban/water)')
Map.centerObject(aoi, zoom=8)
Map

Map(center=[34.95262691883632, -120.23436761217657], controls=(WidgetControl(options=['position', 'transparent…

## 16. Head-to-head

Score dNBR the exact same way I scored the embeddings, including a percentile-matched comparison so it's apples to apples. This is the comparison the whole Phase 1 claim rests on.

Result: the embedding method beat automated dNBR on every metric at every threshold, often 2 to 3x on recall and IoU, with zero training labels. The caveat I keep attached: MTBS perimeters themselves come from a careful, hand-tuned dNBR workflow, so the fair claim is "beat a standard automated dNBR," not "beat a human analyst."

In [17]:
def score_mask(mask):
    """Precision/recall/IoU of a binary mask vs fire_raster, within wildland."""
    mask = mask.rename('m')
    tp = mask.eq(1).And(fire_raster.eq(1)).rename('tp')
    fp = mask.eq(1).And(fire_raster.eq(0)).rename('fp')
    fn = mask.eq(0).And(fire_raster.eq(1)).rename('fn')
    def pc(img, band):
        return (img.select(band).updateMask(wildland)
                   .reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi,
                                 scale=100, maxPixels=1e9, bestEffort=True, tileScale=4)
                   .getNumber(band).getInfo())
    tp_n, fp_n, fn_n = pc(tp,'tp'), pc(fp,'fp'), pc(fn,'fn')
    prec = tp_n/(tp_n+fp_n) if (tp_n+fp_n) else 0
    rec  = tp_n/(tp_n+fn_n) if (tp_n+fn_n) else 0
    iou  = tp_n/(tp_n+fp_n+fn_n) if (tp_n+fp_n+fn_n) else 0
    return prec, rec, iou

print("--- Standard Key & Benson thresholds ---")
for label, t in [('dNBR>=0.10 (low+)', 0.10), ('dNBR>=0.27 (moderate+)', 0.27)]:
    p, r, i = score_mask(dnbr.gt(t))
    print(f"{label}: Precision={p:.3f}  Recall={r:.3f}  IoU={i:.3f}")

print("--- Percentile-matched (head-to-head with embeddings) ---")
dnbr_pcts = dnbr.updateMask(wildland).reduceRegion(
    reducer=ee.Reducer.percentile([97, 98, 99]),
    geometry=aoi, scale=300, maxPixels=1e9, bestEffort=True, tileScale=4)
print("dNBR wildland percentiles:", dnbr_pcts.getInfo())

for key in ['dNBR_p97', 'dNBR_p98', 'dNBR_p99']:
    t = dnbr_pcts.getNumber(key)
    p, r, i = score_mask(dnbr.gt(t))
    print(f"{key}: Precision={p:.3f}  Recall={r:.3f}  IoU={i:.3f}")

--- Standard Key & Benson thresholds ---
dNBR>=0.10 (low+): Precision=0.004  Recall=0.920  IoU=0.004
dNBR>=0.27 (moderate+): Precision=0.011  Recall=0.415  IoU=0.011
--- Percentile-matched (head-to-head with embeddings) ---
dNBR wildland percentiles: {'dNBR_p97': 0.33568225229289156, 'dNBR_p98': 0.3666789349393622, 'dNBR_p99': 0.41377640617751144}
dNBR_p97: Precision=0.013  Recall=0.225  IoU=0.013
dNBR_p98: Precision=0.014  Recall=0.160  IoU=0.013
dNBR_p99: Precision=0.015  Recall=0.087  IoU=0.013


## 17. Double-check the CDL classes

A frequency histogram of which CDL codes actually exist in the area and how common each is. This confirms the mask is grabbing real classes and sanity-checks that grass/pasture (the noisy one) is as common as I expected.

In [18]:
# What CDL codes actually exist in our area, and how common is each?
cdl = (ee.ImageCollection("USDA/NASS/CDL")
       .filterDate('2020-01-01', '2020-12-31')
       .first()
       .select('cropland'))

# Histogram of class codes across the AOI
histogram = cdl.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi,
    scale=100,
    maxPixels=1e9,
    bestEffort=True,
    tileScale=4
).getInfo()

# Print sorted by how common each code is
counts = histogram['cropland']
sorted_codes = sorted(counts.items(), key=lambda x: -x[1])
print("CDL code : pixel count (most common first)")
for code, count in sorted_codes[:25]:
    print(f"  {code} : {int(count):,}")

CDL code : pixel count (most common first)
  152 : 709,222
  176 : 497,966
  143 : 98,855
  142 : 93,307
  121 : 62,929
  69 : 25,062
  122 : 17,948
  123 : 11,980
  61 : 10,787
  21 : 6,305
  195 : 6,096
  131 : 4,189
  215 : 4,148
  111 : 4,099
  36 : 2,912
  75 : 2,215
  204 : 1,872
  37 : 1,714
  190 : 1,497
  76 : 1,458
  28 : 1,241
  72 : 1,186
  124 : 817
  42 : 713
  24 : 710


## 18. Phase 2: from one year to the whole trajectory

Here's where it gets ambitious. Instead of comparing one year to the baseline, I want every flagged pixel's full 2017 to 2025 trajectory, so I can classify *how* it changed, not just whether. First step: confirm which years are available.

In [19]:
import datetime
avail = (ee.ImageCollection(ALPHA_EARTH).filterBounds(aoi)
         .aggregate_array('system:time_start')).getInfo()
years_avail = sorted({datetime.datetime.utcfromtimestamp(t/1000).year for t in avail})
print('AlphaEarth years in AOI:', years_avail)

TRAJ_YEARS = [y for y in range(2017, 2026) if y in years_avail]
print('Trajectory years:', TRAJ_YEARS)

AlphaEarth years in AOI: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Trajectory years: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


## 19. Build the trajectory stack

One band per year, each the distance from the same 2017 to 2019 baseline. By design the baseline years sit near zero, which gives every pixel a flat "normal" stretch to depart from.

(Foreshadowing: measuring everything against a *fixed* baseline is exactly the choice that comes back to bite me at the end.)

In [20]:
def distance_to_baseline(year):
    return cosine_distance(baseline, load_embedding_year(year)).rename(f'd_{year}')

traj = ee.Image.cat([distance_to_baseline(y) for y in TRAJ_YEARS])  # one band per year, in TRAJ_YEARS order

## 20. Turn each trajectory into shape features

The interpretable bits: when the disturbance peaked, how abrupt the onset was (one big jump vs a slow slide), and how far it has subsided since. The neat part is that because the years are consecutive, the peak year is just the peak index plus 2017, so I never have to index an array by a per-pixel value.

In [21]:
n = len(TRAJ_YEARS)
arr = traj.toArray()  # 1-D array image, ordered like TRAJ_YEARS

# Peak disturbance and when it happened
peak      = arr.arrayReduce(ee.Reducer.max(), [0]).arrayGet([0]).rename('peak')
peak_idx  = arr.arrayArgmax().arrayGet([0]).rename('peak_idx')          # 0-based
peak_year = peak_idx.add(TRAJ_YEARS[0]).rename('peak_year')

# Abruptness: biggest single-year rise, as a fraction of the peak
later   = arr.arraySlice(0, 1, n)
earlier = arr.arraySlice(0, 0, n - 1)
max_jump    = later.subtract(earlier).arrayReduce(ee.Reducer.max(), [0]).arrayGet([0]).rename('max_jump')
abrupt_frac = max_jump.divide(peak).rename('abrupt_frac')              # ~1 = one-year crash, ~0.25 = slow slide

# Outcome: how far it has subsided from peak by the last year
d_last        = traj.select(f'd_{TRAJ_YEARS[-1]}').rename('d_last')
recovery_frac = peak.subtract(d_last).divide(peak).rename('recovery_frac')   # ~1 recovered, ~0 still elevated
years_since   = ee.Image.constant(TRAJ_YEARS[-1]).subtract(peak_year).rename('years_since')

## 21. Classify the signatures

Two independent axes, because "abrupt vs gradual vs recovering" is really two questions wearing one label: how the change arrived (onset) and where it ended up (outcome). I keep both as separate bands and derive a single four-class signature for the legend: abrupt loss, gradual decline, recovering, and too recent to judge.

In [22]:
ABRUPT_CUT, RECOVER_CUT, MIN_YEARS = 0.50, 0.40, 2

pk = peak.updateMask(wildland).reduceRegion(
    reducer=ee.Reducer.percentile([95, 97, 98, 99]),
    geometry=aoi, scale=300, maxPixels=1e9, bestEffort=True, tileScale=4)
thr = ee.Number(pk.get('peak_p97'))     # p97 default, swap key to retune
disturbed = peak.gt(thr)

# Axis 1: onset (1 abrupt, 2 gradual)
onset = (ee.Image(0)
    .where(disturbed.And(abrupt_frac.gte(ABRUPT_CUT)), 1)
    .where(disturbed.And(abrupt_frac.lt(ABRUPT_CUT)), 2).rename('onset'))

# Axis 2: outcome (1 recovering, 2 persistent, 3 too recent to judge)
outcome = (ee.Image(0)
    .where(disturbed.And(years_since.lt(MIN_YEARS)), 3)
    .where(disturbed.And(years_since.gte(MIN_YEARS)).And(recovery_frac.gte(RECOVER_CUT)), 1)
    .where(disturbed.And(years_since.gte(MIN_YEARS)).And(recovery_frac.lt(RECOVER_CUT)), 2).rename('outcome'))

# Combined 4-class legend: 1 abrupt loss, 2 gradual decline, 3 recovering, 4 recent
signature = (ee.Image(0)
    .where(disturbed.And(outcome.eq(3)), 4)
    .where(disturbed.And(outcome.eq(1)), 3)
    .where(disturbed.And(outcome.eq(2)).And(onset.eq(1)), 1)
    .where(disturbed.And(outcome.eq(2)).And(onset.eq(2)), 2)
    .rename('signature')).updateMask(wildland)

features = ee.Image.cat([peak, peak_year, max_jump, abrupt_frac,
                         recovery_frac, years_since, onset, outcome, signature])

## 22. Look at the typed map

The signature map: red abrupt, amber gradual, green recovering, blue recent. This is the version that, if it had held up, would have been worth shipping.

In [23]:
import geemap
m = geemap.Map()
m.centerObject(aoi, 9)
# 1 abrupt=red, 2 gradual=amber, 3 recovering=green, 4 recent=blue
m.addLayer(signature.selfMask(), {'min': 1, 'max': 4,
           'palette': ['d73027', 'fdae61', '1a9850', '4575b4']}, 'Signature')
m.addLayer(fires_2020, {'color': '000000'}, 'MTBS 2020', False)
m

Map(center=[34.952626918836316, -120.23436761217651], controls=(WidgetControl(options=['position', 'transparen…

## 23. Validate the typing, not just the detection

The real test: do the signatures line up with reality? I compare the signature mix inside known fire perimeters against the wildland outside them. Fires should skew abrupt, with some recovering for fast-regrowing grass.

It half-worked. Abrupt loss was about 5x more concentrated inside fires than outside, which is the right direction. But "recent" swallowed 83% of the wildland outside fires, which is not a thing that physically happens. That number is what kicked off the investigation below.

In [24]:
start_ms = ee.Date(f'{TRAJ_YEARS[0]}-01-01').millis()
end_ms   = ee.Date(f'{TRAJ_YEARS[-1] + 1}-01-01').millis()
fires_all = (frap_all.filter(ee.Filter.bounds(aoi))
             .filter(ee.Filter.rangeContains('Ig_Date', start_ms, end_ms)))
fire_all_r = ee.Image(0).paint(fires_all, 1).rename('fire').clip(aoi).unmask(0)

NAMES = {'1': 'abrupt loss', '2': 'gradual decline', '3': 'recovering', '4': 'recent'}
def composition(region_mask, label):
    hist = ee.Dictionary(signature.selfMask().updateMask(region_mask).reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(), geometry=aoi,
        scale=100, maxPixels=1e9, bestEffort=True, tileScale=16).get('signature')).getInfo()
    total = sum(hist.values()) or 1
    print(f"\n{label}")
    for k in sorted(hist):
        print(f"  {NAMES.get(k, k):16s} {100 * hist[k] / total:5.1f}%  ({int(hist[k])} px)")

composition(fire_all_r.eq(1), 'Inside known fire perimeters (2017 to 2025):')
composition(fire_all_r.eq(0).And(wildland), 'Wildland outside fires:')


Inside known fire perimeters (2017 to 2025):
  abrupt loss       21.7%  (1252 px)
  gradual decline    4.8%  (275 px)
  recovering        25.9%  (1491 px)
  recent            47.7%  (2748 px)

Wildland outside fires:
  abrupt loss        4.4%  (2137 px)
  gradual decline    2.2%  (1085 px)
  recovering        10.3%  (5029 px)
  recent            83.1%  (40697 px)


## 24. Cache the heavy layer

Every diagnostic re-runs the entire computation graph from scratch (baseline mean, nine years of distance, all the array math), and Earth Engine started refusing the bigger queries. So I exported the feature stack to an asset once, to run everything downstream against a flat, cheap layer instead of rebuilding the monster each time.

In [25]:
task = ee.batch.Export.image.toAsset(
    image=features,
    description='verdant_drift_features',
    assetId='projects/ee-jerrodlessel/assets/verdant_drift_features',
    region=aoi, scale=10, maxPixels=1e10)
task.start()
print('Export started. Monitor with: ee.batch.Task.list() or the Tasks tab.')

Export started. Monitor with: ee.batch.Task.list() or the Tasks tab.


## 25. Reload from the asset

Point everything downstream at the exported asset. Cheap to query, no more capacity errors.

In [ ]:
feat = ee.Image('projects/ee-XXXXXXXXX/assets/verdant_drift_features')
signature     = feat.select('signature')
recovery_frac = feat.select('recovery_frac')
# ...validation, viz, and Phase 3 all read from `feat` cheaply now

## 26. The smoking gun

This is the diagnostic that ended the project. I histogrammed the peak-disturbance year across all flagged pixels. If the detector were finding real disturbance, peak years would spread across the record.

Instead, the overwhelming majority "peaked" in 2025, the most recent year. Around 72% of flagged pixels, all peaking at once, in the last year. That is not how landscapes disturb. Something was wrong with the method, not the land.

In [ ]:
feat = ee.Image('projects/ee-XXXXXXXXXX/assets/verdant_drift_features')
disturbed = feat.select('signature').gte(1)
py = ee.Dictionary(feat.select('peak_year').updateMask(disturbed).reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(), geometry=aoi,
    scale=100, maxPixels=1e9, bestEffort=True, tileScale=8).get('peak_year')).getInfo()
for y in sorted(py): print(f"  {y}: {int(py[y])} px")

  2017: 67 px
  2018: 193 px
  2019: 51 px
  2020: 1882 px
  2021: 2418 px
  2022: 7474 px
  2023: 8337 px
  2024: 7610 px
  2025: 73888 px


## 27. Task-status helper

A little helper to check whether the asset export had actually finished before I tried to use it. (It hadn't, the first time. Patience.)

In [28]:
for t in ee.batch.Task.list()[:5]:
    s = t.status()
    print(s.get('description'), '->', s.get('state'))

verdant_drift_features -> FAILED
verdant_drift_features -> COMPLETED


## 28. Following the thread: per-year means

To see what 2025 was doing, I averaged the distance-from-baseline across all wildland, year by year. The shape told the story: near-zero through the baseline, a clear drought bump in 2021 to 2022, easing back in 2023 to 2024, then 2025 spiking to the highest value in the whole record, landscape-wide.

Landscape-wide is the key phrase. Whole-landscape shifts aren't disturbance. They're the year itself being different.

In [29]:
means = traj.updateMask(wildland).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi,
    scale=300, maxPixels=1e9, bestEffort=True, tileScale=16).getInfo()
for y in TRAJ_YEARS:
    print(f"  {y}: mean d = {means[f'd_{y}']:.4f}")

  2017: mean d = 0.0233
  2018: mean d = 0.0270
  2019: mean d = 0.0124
  2020: mean d = 0.0337
  2021: mean d = 0.0629
  2022: mean d = 0.0601
  2023: mean d = 0.0464
  2024: mean d = 0.0341
  2025: mean d = 0.0867


## 29. Confirming it: the whole distribution moved

The final nail. I broke 2023 to 2025 into percentiles. In 2025, even the calmest pixels (the 10th percentile) were further from baseline than a *median* pixel had been the year before. The entire distribution lifted, floor and all.

That confirmed the flaw: distance from a fixed 2017 to 2019 baseline doesn't only measure "did this pixel get disturbed." It also measures "was this year climatically different from the baseline window." A dry year shifts everything, so my detector was, in part, an time-expensive drought thermometer.

The fix exists (normalize each year against its own landscape). But the feature I actually wanted, a recovery metric, is measured at the end of the record, which was 2025, the single most poisoned year. Building it would have produced a confident, good-looking map that was wrong. So I stopped here. The full reasoning is in the postmortem writeup. 🌲

In [30]:
for y in [2023, 2024, 2025]:
    p = traj.select(f'd_{y}').updateMask(wildland).reduceRegion(
        reducer=ee.Reducer.percentile([10, 50, 90, 99]),
        geometry=aoi, scale=300, maxPixels=1e9, bestEffort=True, tileScale=16).getInfo()
    print(y, {k: round(v, 4) for k, v in p.items()})

2023 {'d_2023_p10': 0.0177, 'd_2023_p50': 0.0371, 'd_2023_p90': 0.0878, 'd_2023_p99': 0.1776}
2024 {'d_2024_p10': 0.0137, 'd_2024_p50': 0.0253, 'd_2024_p90': 0.0644, 'd_2024_p99': 0.1856}
2025 {'d_2025_p10': 0.0372, 'd_2025_p50': 0.0762, 'd_2025_p90': 0.1465, 'd_2025_p99': 0.2677}
